# Day 18 · NULL — แผนที่ของความว่าง

**#30DayMapChallenge 2026 · 18 พ.ย. 2569** · โจทย์บนหน้าเว็บมีคำเดียวว่า **"NULL"**
ไม่มีประโยคอธิบาย (ดึงอ่านสดสองครั้งเพื่อยืนยัน) — โจทย์จึงเป็น NULL ด้วยตัวมันเอง
แผ่นนี้เขียนและเผยแพร่คืนวันที่ 19 ต่อเช้าวันที่ 20 ช้าหนึ่งวันตามจริงที่บอกไว้ในโพสต์

วัดความว่างสามชนิดของทะเบียนแหล่งท้องฟ้ามืด 83 จุด × 19 ฟิลด์ (1,577 เซลล์)

| ชนิด | ตัวอย่างที่เจอ |
|---|---|
| NULL จริง | `tourism_types` ว่าง 60/83 · `events_held` 50/83 · รวมทั้งตาราง 219 เซลล์ (13.89%) |
| NULL ที่แกล้งเป็นข้อมูล | `registration_year` เติมครบแต่ 82/83 เป็น *ช่วงปี* (14 รูปแบบ, ตัวคั่น 4 แบบ) · `sky_quality_sqm` ปนจำนวนเต็ม 77 กับทศนิยม 6 · `contact` มีค่าซ้ำ 2 ค่าใน 4 จุด |
| NULL ที่แผนที่สร้าง | 4 จุดนอกเส้นขอบประเทศ · 7 จุดไม่มี hex ในกริด Day 3 (5 จุดถ้าเทียบกริด NE 10m) · 3 จุดรอบตัวไม่มี way ใน OSM |

**จุดสำคัญของการอ่านโค้ด**
- ต้องอ่าน CSV ด้วย `keep_default_na=False` แล้วเทียบกับสตริงว่าง (หลัง strip) —
  ไม่งั้น pandas จะแปลงค่าว่างเป็น NaN เองและนับไม่ตรง
- มี assert ข้ามวัน: ตัวเลข 4 / 7 / 5 ต้องตรงกับ `day14_stats.json` (ซึ่งบันทึกไว้ *สอง* นิยาม)
  assert นี้จับความเข้าใจผิดของผมได้จริงตอนเขียน
- ทุกตัวเลขในโพสต์อ่านจาก `day18_null_sites.csv` / `day18_by_holder.csv` / `day18_stats.json`

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 18 · NULL — โจทย์บนหน้า 30daymapchallenge.com วันที่ 18 พ.ย. 2569 คือคำว่า "NULL"
โดย *ไม่มีประโยคอธิบายเลย* (ดึงหน้าเว็บอ่านสด ยืนยันสองครั้ง) — โจทย์จึงเป็น NULL ด้วยตัวมันเอง

แผ่นนี้วัดความว่างสามชนิดของทะเบียนแหล่งท้องฟ้ามืด NARIT (83 จุด × 19 ฟิลด์)
  ๑ NULL จริง      — ฟิลด์ที่เว้นว่าง (นับต่อคอลัมน์ ต่อจุด ตามผู้ถือครอง)
  ๒ NULL ที่แกล้งเป็นข้อมูล — ฟิลด์ที่เติมครบ 100% แต่ไม่ให้ข้อเท็จจริงตามที่ชื่อคอลัมน์สัญญา
  ๓ NULL ที่แผนที่สร้าง — จุดทะเบียนที่หายไปจากแผนที่มาตรฐาน (ต่อจาก Day 14 และ Day 16)

ทุกตัวเลขอ่านจาก day18_null_sites.csv / day18_stats.json เท่านั้น
"""
import os, sys, json, time, re, collections
import numpy as np, pandas as pd, geopandas as gpd
import h3
from shapely.geometry import Point
from shapely.validation import make_valid
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Wedge, Circle
from matplotlib.collections import PolyCollection

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06")
OUT = os.path.join(WORK, "out", "day18")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day18_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

sys.path.insert(0, os.path.join(WORK, "tools"))
from guards import guard_bbox, guard_crs, guard_variance, lint_thai, guard_labels

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
SHORT = {"เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคล": "ที่ดินส่วนบุคคล",
         "อุทยานท้องฟ้ามืด": "อุทยานท้องฟ้ามืด",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ชานเมือง": "ชานเมือง",
         "ชุมชนอนุรักษ์ท้องฟ้ามืด": "ชุมชน",
         "อุทยานแห่งชาติ": "อุทยานแห่งชาติ",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคลเขตชุมชน": "ส่วนบุคคล+ชุมชน"}

## ตอน 1) อ่านทะเบียนแบบห้ามแปลงค่าว่าง

In [ ]:
# ---------------------------------------------------------------- 1. อ่านทะเบียนแบบห้ามแปลงค่าว่าง
df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"), dtype=str, keep_default_na=False)
df["name"] = df["name"].astype(str).str.strip()
assert len(df) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(df)}"
COLS = list(df.columns)
N_COL = len(COLS)
blank = df.apply(lambda c: c.astype(str).str.strip() == "")
per_site_blank = blank.sum(axis=1).values
per_col_blank = blank.sum(axis=0)
holder = df["type"].astype(str).str.strip().map(lambda t: SHORT.get(t, t)).values
region = df["region_th"].astype(str).str.strip().values

SITES = np.column_stack([df.longitude.values.astype(float), df.latitude.values.astype(float)])
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")
log(f"ทะเบียน {len(df)} จุด × {N_COL} ฟิลด์ · ฟิลด์ว่างต่อจุด: ต่ำสุด {per_site_blank.min()} · "
    f"มัธยฐาน {np.median(per_site_blank):.1f} · สูงสุด {per_site_blank.max()}")
sparse = [(c, int(per_col_blank[c])) for c in COLS if per_col_blank[c] > 0]
sparse.sort(key=lambda x: -x[1])
log("คอลัมน์ที่มีค่าว่าง: " + " · ".join(f"{c} {n}/83 ({100*n/83:.1f}%)" for c, n in sparse))
log("คอลัมน์ที่เติมครบ 100%: " + ", ".join([c for c in COLS if per_col_blank[c] == 0]))

## ตอน 2) NULL ที่แกล้งเป็นข้อมูล

In [ ]:
# --------------------------------------------- 2. NULL ที่แกล้งเป็นข้อมูล
ry = df["registration_year"].astype(str).str.strip()
RY = dict(distinct=int(ry.nunique()),
          counts={k: int(v) for k, v in ry.value_counts().items()},
          is_range=int(ry.str.contains(r"[–-]").sum()),
          is_single_year=int((~ry.str.contains(r"[––-]")).sum()),
          separator_variants={
              "en dash": int(ry.str.contains("\u2013").sum()),
              "hyphen": int(ry.str.contains(r"\d\s*-\s*\d").sum()),
              "คำว่า และ": int(ry.str.contains("และ").sum()),
              "จุลภาค": int(ry.str.contains(",").sum())},
          mentions_2570_or_later=int(ry.str.contains("2570|2571").sum()))
log(f"registration_year: ไม่ซ้ำ {RY['distinct']} ค่า · เป็นช่วง {RY['is_range']}/83 · "
    f"เป็นปีเดี่ยว {RY['is_single_year']} · ตัวคั่น 4 แบบ {RY['separator_variants']}")

sq = df["sky_quality_sqm"].astype(str).str.strip()
SQM = dict(distinct=int(sq.nunique()),
           integer_form=int(sq.str.fullmatch(r"\d+").sum()),
           decimal_form=int(sq.str.contains(r"\.").sum()),
           modal_value=str(sq.value_counts().index[0]),
           modal_count=int(sq.value_counts().iloc[0]),
           values=sorted(sq.unique().tolist()))
raw = df["sky_quality_raw"].astype(str).str.strip()
RAWQ = dict(distinct=int(raw.nunique()),
            all_start_with_sqm=int(raw.str.startswith("SQM").sum()),
            example=str(raw.value_counts().index[0]))
log(f"sky_quality_sqm: ไม่ซ้ำ {SQM['distinct']} ค่า · จำนวนเต็ม {SQM['integer_form']} จุด "
    f"ทศนิยม {SQM['decimal_form']} จุด · ค่าที่พบมากที่สุด {SQM['modal_value']} ×{SQM['modal_count']}")

ct = df["contact"].astype(str).str.strip()
dup_vals = ct[ct.duplicated(keep=False)].unique().tolist()
CONTACT = dict(distinct=int(ct.nunique()), duplicated_values=int(len(dup_vals)),
               sites_involved=int(ct.isin(dup_vals).sum()),
               examples=[{"value_head": str(v)[:38], "sites": int((ct == v).sum()),
                          "names": df.name[ct == v].tolist()} for v in dup_vals])
log(f"contact: ไม่ซ้ำ {CONTACT['distinct']}/83 · ค่าที่ใช้ซ้ำ {CONTACT['duplicated_values']} ค่า "
    f"ครอบคลุม {CONTACT['sites_involved']} จุด")

## ตอน 3) NULL ที่แผนที่สร้าง (คำนวณใหม่ + เทียบ Day 14/16)

In [ ]:
# ------------------------------------- 3. NULL ที่แผนที่สร้าง (คำนวณใหม่ + เทียบ Day 14/16)
ne = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
guard_crs(TH.bounds, "thailand-outline", expect="degrees")
PARTS = list(TH.geoms)
g3 = gpd.read_file(os.path.join(REPO, "posts/day03-polygons/day03_hex_polygons.geojson"))
grid_cells = set(str(c) for c in g3.cell.values)
outside_outline = [str(n) for n in df.name.values[
    [not TH.contains(Point(float(lo), float(la))) for la, lo in
     zip(df.latitude.values.astype(float), df.longitude.values.astype(float))]]]
no_own_cell = sorted({str(df.name.iloc[i]) for i in range(83)
                      if h3.latlng_to_cell(float(df.latitude.iloc[i]), float(df.longitude.iloc[i]), 5)
                      not in grid_cells})
# เทียบกับกริด res 5 ที่เติมจาก Natural Earth 10m (คนละชุดกับกริด Day 3 ที่สร้างจาก GADM)
def _ring(coords):
    out = []
    for c in coords:
        ll = (round(float(c[1]), 7), round(float(c[0]), 7))
        if not out or out[-1] != ll:
            out.append(ll)
    if len(out) > 1 and out[0] == out[-1]:
        out.pop()
    return out
HPS = []
for p in PARTS:
    loops = [_ring(p.exterior.coords)] + [_ring(i.coords) for i in p.interiors]
    loops = [l for l in loops if len(l) >= 3]
    if loops:
        HPS.append(h3.LatLngPoly(loops[0], *loops[1:]))
NE_FILL = set(str(c) for c in h3.polygon_to_cells(h3.LatLngMultiPoly(*HPS), 5))
no_own_cell_ne = sorted({str(df.name.iloc[i]) for i in range(83)
                         if h3.latlng_to_cell(float(df.latitude.iloc[i]),
                                              float(df.longitude.iloc[i]), 5) not in NE_FILL})
d16 = pd.read_csv(os.path.join(REPO, "posts/day16-collaborative-map/day16_sites_osm.csv"))
zero_map = sorted(d16.name[d16.ways == 0].tolist())
MAPNULL = dict(outside_outline=sorted(outside_outline), n_outside=len(outside_outline),
               no_own_cell=no_own_cell, n_no_own_cell=len(no_own_cell),
               no_own_cell_reference="กริด Day 3 (GADM, 1,738 เซลล์)",
               no_own_cell_vs_ne10m_fill=no_own_cell_ne,
               n_no_own_cell_vs_ne10m_fill=len(no_own_cell_ne),
               ne10m_fill_cells=len(NE_FILL), day3_grid_cells=len(grid_cells),
               zero_way_sites=zero_map, n_zero_way=len(zero_map))
multi_null = ((set(MAPNULL["outside_outline"]) & set(MAPNULL["no_own_cell"])) |
              (set(MAPNULL["outside_outline"]) & set(zero_map)) |
              (set(MAPNULL["no_own_cell"]) & set(zero_map)))
MAPNULL["sites_null_in_two_or_more_senses"] = sorted(multi_null)
MAPNULL["n_multi"] = len(multi_null)
log(f"นอกเส้นขอบประเทศ {len(outside_outline)} จุด · ไม่มี hex ในกริด Day 3 {len(no_own_cell)} จุด "
    f"(ถ้าเทียบกับกริด NE 10m: {len(no_own_cell_ne)} จุด) · รอบจุดไม่มี way เลย {len(zero_map)} จุด · "
    f"ว่างพร้อมกัน >=2 ความหมาย {len(multi_null)} จุด")
log("  จุดที่ว่างซ้ำซ้อน: " + ", ".join(sorted(multi_null)))

## ตอน 4) สถิติตามผู้ถือครอง / ภูมิภาค

In [ ]:
# ------------------------------------- 4. สถิติตามผู้ถือครอง / ภูมิภาค
by_holder = (pd.DataFrame({"holder": holder, "blank": per_site_blank})
             .groupby("holder").blank.agg(["size", "min", "median", "max"]).reset_index())
by_holder["mean"] = (pd.DataFrame({"holder": holder, "blank": per_site_blank})
                     .groupby("holder").blank.mean().values)
by_region = (pd.DataFrame({"region": region, "blank": per_site_blank})
             .groupby("region").blank.agg(["size", "median", "max"]).reset_index())
log("ฟิลด์ว่างต่อจุดตามผู้ถือครอง: " + " · ".join(
    f"{r.holder} {r.median:.1f} (n={int(r.size)})" for r in by_holder.itertuples()))

## ตอน 5) ตารางต่อจุด

In [ ]:
# ------------------------------------- 5. ตารางต่อจุด
rows = []
for i in range(83):
    rows.append(dict(idx=i, name=df.name.iloc[i], province=str(df.province.iloc[i]).strip(),
                     holder=holder[i], region=region[i],
                     blank_fields=int(per_site_blank[i]),
                     filled_fields=int(N_COL - per_site_blank[i]),
                     completeness_pct=round(100 * (N_COL - per_site_blank[i]) / N_COL, 1),
                     blank_columns=";".join([c for c in COLS if blank[c].iloc[i]]),
                     registration_year=ry.iloc[i], sky_quality_sqm=sq.iloc[i],
                     sqm_precision=("ทศนิยม" if "." in sq.iloc[i] else "จำนวนเต็ม"),
                     contact_duplicated=bool(ct.iloc[i] in dup_vals),
                     outside_outline=bool(str(df.name.iloc[i]) in outside_outline),
                     no_own_cell=bool(str(df.name.iloc[i]) in no_own_cell),
                     zero_way_map=bool(str(df.name.iloc[i]) in zero_map)))
S = pd.DataFrame(rows)
assert S.blank_fields.max() <= N_COL
guard_variance(S.blank_fields.values.astype(float), "blank-per-site", min_spread_km=0)

# ================================================== ภาพ 1 — แผนที่ของความไม่มี
def draw_hero():
    fig = plt.figure(figsize=(15.4, 8.6), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.02, 0.115, 0.66, 0.775]); ax.set_facecolor("#05080f")
    ax.add_collection(PolyCollection([list(p.exterior.coords) for p in PARTS],
                                     facecolor="#101a2c", edgecolor="#4a6288", lw=0.9))
    R = 0.135
    for _, r in S.iterrows():
        frac = r.filled_fields / N_COL
        if r.outside_outline or r.zero_way_map:
            ax.add_patch(Circle((r_lon(r), r_lat(r)), R * 1.22, fill=False,
                                edgecolor=WARM, lw=1.3, linestyle="--", zorder=6))
        for k in range(12):
            a0, a1 = k * 30.0, k * 30.0 + 30.0 - 0.6
            on = (k / 12.0) < frac
            ax.add_patch(Wedge((r_lon(r), r_lat(r)), R, a0, a1,
                               facecolor=(ACCENT if on else "#1b2740"),
                               edgecolor=(INK if on else "#33415e"), lw=0.35, zorder=4))
        ax.add_patch(Circle((r_lon(r), r_lat(r)), R * 0.32, facecolor="#05080f",
                            edgecolor=INK, lw=0.5, zorder=5))
    ax.set_xlim(97.2, 106.6); ax.set_ylim(5.2, 20.9); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(T("วงแหวน 12 ช่องต่อจุด = 19 ฟิลด์ของทะเบียน · ช่องสว่างคือฟิลด์ที่มีข้อความ"),
                 color=INK, fontsize=12.5, pad=10)
    txt = fig.add_axes([0.715, 0.10, 0.275, 0.82]); txt.axis("off")
    lines = [(T("ฟิลด์ทั้งหมด"), f"{N_COL} × 83"),
             (T("จุดที่ว่าง 0 ฟิลด์"), f"{int((per_site_blank == 0).sum())}"),
             (T("ฟิลด์ว่างต่อจุด · มัธยฐาน"), f"{np.median(per_site_blank):.1f}"),
             (T("ฟิลด์ว่างต่อจุด · สูงสุด"), f"{int(per_site_blank.max())}"),
             ("", ""),
             (T("tourism_types ว่าง"), f"{int(per_col_blank['tourism_types'])}/83"),
             (T("events_held ว่าง"), f"{int(per_col_blank['events_held'])}/83"),
             (T("facilities ว่าง"), f"{int(per_col_blank['facilities'])}/83"),
             (T("details_note ว่าง"), f"{int(per_col_blank['details_note'])}/83"),
             (T("address ว่าง"), f"{int(per_col_blank['address'])}/83"),
             ("", ""),
             (T("registration_year ไม่ซ้ำ"), f"{RY['distinct']} แบบ"),
             (T("  ที่เป็นช่วง (ไม่ใช่ปี)"), f"{RY['is_range']}/83"),
             (T("sky_quality_sqm ไม่ซ้ำ"), f"{SQM['distinct']} ค่า"),
             (T("  เขียนเป็นทศนิยม"), f"{SQM['decimal_form']} จุด"),
             (T("contact ที่ใช้ซ้ำ"), f"{CONTACT['duplicated_values']} ค่า / {CONTACT['sites_involved']} จุด"),
             ("", ""),
             (T("นอกเส้นขอบประเทศ"), f"{MAPNULL['n_outside']} จุด"),
             (T("ไม่มี hex ในกริด Day 3"), f"{MAPNULL['n_no_own_cell']} จุด"),
             (T("  ถ้านับกริด NE 10m"), f"{MAPNULL['n_no_own_cell_vs_ne10m_fill']} จุด"),
             (T("รอบจุดไม่มี way ใน OSM"), f"{MAPNULL['n_zero_way']} จุด"),
             (T("ว่างซ้ำซ้อน >=2 แบบ"), f"{MAPNULL['n_multi']} จุด")]
    y = 0.98
    for k, v in lines:
        if not k:
            y -= 0.026; continue
        txt.text(0.0, y, k, color=MUTED, fontsize=10.2, va="top")
        txt.text(1.0, y, v, color=ACCENT, fontsize=10.2, va="top", ha="right")
        y -= 0.0432
    fig.text(0.02, 0.062, T("จุดที่วาดเป็นวงประสีส้ม = จุดที่ *แผนที่มาตรฐาน* ไม่มีอยู่จริง "
                            "(อยู่นอกเส้นขอบประเทศ หรือรอบจุดไม่มีวัตถุใน OpenStreetMap เลย)"),
             color=WARM, fontsize=10.5)
    fig.text(0.02, 0.034, T("โจทย์วันนี้มีคำเดียวว่า NULL และไม่มีประโยคอธิบาย — "
                            "แผ่นนี้จึงวาดเฉพาะสิ่งที่ *ไม่มี* ในทะเบียน 83 จุด"),
             color=MUTED, fontsize=10)
    fig.suptitle(T("แผนที่ของความว่าง · ทะเบียนแหล่งท้องฟ้ามืด 83 จุด 19 ฟิลด์"),
                 color=INK, fontsize=15.5, y=0.985)
    p = os.path.join(OUT, "day18_null.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

def r_lon(r):
    return float(df.longitude.iloc[int(r.idx)])
def r_lat(r):
    return float(df.latitude.iloc[int(r.idx)])

# ================================================== ภาพ 2 — matrix ของความว่าง
def draw_matrix():
    order = np.argsort(-per_site_blank, kind="stable")
    M = blank.values[order][:, [c for c in range(N_COL)]]
    fig, ax = plt.subplots(figsize=(14.8, 7.4), dpi=150)
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    ax.imshow(M, aspect="auto", cmap="gray_r", vmin=0, vmax=1)
    ax.set_xticks(range(N_COL))
    ax.set_xticklabels(COLS, rotation=42, ha="right", color=INK, fontsize=9.5)
    ax.set_yticks([])
    ax.set_ylim(M.shape[0] - 0.5, -0.5)
    for j, c in enumerate(COLS):
        n = int(per_col_blank[c])
        if n:
            ax.text(j, -1.6, f"{n}", color=WARM, fontsize=9, ha="center")
    ax.set_title(T("ดำ = ฟิลด์ที่เว้นว่าง · ขาว = มีข้อความ (83 จุดเรียงจากว่างมากไปน้อย, 19 คอลัมน์)"),
                 color=INK, fontsize=12.5, pad=16)
    ax.tick_params(colors=INK, labelsize=9)
    for s in ax.spines.values(): s.set_color("#33415e")
    fig.text(0.02, 0.02, T("ตัวเลขสีส้มเหนือคอลัมน์คือจำนวนจุดที่เว้นฟิลด์นั้น "
                           "(ห้าคอลัมน์ขวามือว่างเกินครึ่งของทะเบียน)"),
             color=MUTED, fontsize=10)
    fig.suptitle(T("NULL มีรูปแบบ ไม่ใช่เรื่องสุ่ม"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day18_matrix.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

# ================================================== ภาพ 3 — NULL ที่แกล้งเป็นข้อมูล
def draw_fake():
    fig = plt.figure(figsize=(15.6, 7.2), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.235, 0.115, 0.335, 0.70]); ax.set_facecolor("#111c2b")
    vals = sorted(RY["counts"].items(), key=lambda x: -x[1])[:10]
    lab = [v[0].replace("พ.ศ.", "").replace(" ", "") for v in vals]
    cnt = [v[1] for v in vals]
    ax.barh(range(len(lab)), cnt, color=COOL, edgecolor=BG, lw=0.3, height=0.66)
    for i, c in enumerate(cnt):
        ax.text(c + 0.35, i, f"{c}", va="center", color=INK, fontsize=10)
    ax.set_yticks(range(len(lab)))
    ax.set_yticklabels(lab, color=INK, fontsize=9.5)
    ax.invert_yaxis(); ax.set_xlim(0, max(cnt) * 1.2)
    ax.set_xlabel(T("จำนวนจุด (จาก 83)"), color=INK)
    ax.set_title(T("คอลัมน์ชื่อ registration_year" + chr(10) +
                   f"แต่ {RY['is_range']}/83 เก็บ *ช่วงปี* ไม่ใช่ปี ({RY['distinct']} รูปแบบ)"),
                 color=INK, fontsize=12)
    ax.tick_params(colors=INK, labelsize=9.5)
    ax.grid(color="#26364f", lw=0.5, alpha=0.5, axis="x")
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): ax.spines[sp].set_color("#33415e")

    ax2 = fig.add_axes([0.635, 0.115, 0.115, 0.70]); ax2.set_facecolor("#111c2b")
    ax2.bar([0, 1], [SQM["integer_form"], SQM["decimal_form"]],
            color=[TEAL, WARM], width=0.6, edgecolor=BG, lw=0.4)
    for i, v in enumerate([SQM["integer_form"], SQM["decimal_form"]]):
        ax2.text(i, v + 1.5, f"{v}", ha="center", color=INK, fontsize=11)
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels([T("จำนวนเต็ม"), T("ทศนิยม")], color=INK, fontsize=10)
    ax2.set_ylim(0, max(SQM["integer_form"], SQM["decimal_form"]) * 1.25)
    ax2.set_ylabel(T("จำนวนจุด"), color=INK, fontsize=10)
    ax2.set_title(T("sky_quality_sqm" + chr(10) + f"{SQM['distinct']} ค่า สองระดับ" + chr(10) +
                    "ความละเอียด"),
                  color=INK, fontsize=11)
    ax2.tick_params(colors=INK, labelsize=9.5)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.5, axis="y")
    for sp in ("top", "right"): ax2.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): ax2.spines[sp].set_color("#33415e")

    txt = fig.add_axes([0.785, 0.115, 0.205, 0.70]); txt.axis("off"); txt.set_facecolor("#111c2b")
    txt.text(0.0, 1.00, T("contact ที่ไม่ unique"), color=INK, fontsize=12, va="top")
    y = 0.90
    for ex in CONTACT["examples"]:
        txt.text(0.0, y, ex["value_head"] + "…", color=MUTED, fontsize=10, va="top")
        y -= 0.075
        txt.text(0.0, y, f"ใช้ใน {ex['sites']} จุด:", color="#cfe0f0", fontsize=10, va="top")
        y -= 0.075
        for nm in ex["names"]:
            txt.text(0.03, y, "· " + nm[:26], color="#cfe0f0", fontsize=9.5, va="top")
            y -= 0.065
        y -= 0.03
    txt.text(0.0, 0.16, T("ฟิลด์ที่ควรมีจุดละหนึ่งค่า" + chr(10) +
                          f"แต่มี {CONTACT['distinct']} ค่าไม่ซ้ำ จาก {len(df)} จุด"),
             color=MUTED, fontsize=10, va="top")
    txt.text(0.0, 0.045, T("sky_quality_raw เป็นข้อความอิสระ" + chr(10) +
                           f"{RAWQ['distinct']} แบบ เช่น {RAWQ['example']}"),
             color=MUTED, fontsize=10, va="top")
    fig.suptitle(T("เติมครบ 100% ไม่ได้แปลว่ามีข้อมูล"), color=INK, fontsize=15, y=0.98)
    p = os.path.join(OUT, "day18_fake_nulls.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_hero(), draw_matrix(), draw_fake()

## ตอน 6) artifact

In [ ]:
# ---------------------------------------------------------------- 6. artifact
S.round(3).to_csv(os.path.join(OUT, "day18_null_sites.csv"), index=False)
by_holder.round(3).to_csv(os.path.join(OUT, "day18_by_holder.csv"), index=False)
STATS = dict(
    theme="NULL",
    theme_note="หน้า 30daymapchallenge.com วันที่ 18-11-2026 ให้แค่คำว่า NULL ไม่มีประโยคอธิบาย "
               "(ดึงอ่านสดสองครั้งเพื่อยืนยัน) — โจทย์จึงเป็น NULL ด้วยตัวมันเอง",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    posted_late="เขียนและเผยแพร่คืนวันที่ 19 พ.ย. สำหรับโจทย์วันที่ 18 พ.ย. (ช้าหนึ่งวัน)",
    register=dict(sites=int(len(df)), columns=N_COL,
                  cells_total=int(len(df) * N_COL),
                  cells_blank=int(blank.values.sum()),
                  blank_share_pct=round(100 * blank.values.sum() / (len(df) * N_COL), 2)),
    column_missing=[{"column": c, "blank": int(per_col_blank[c]),
                     "blank_pct": round(100 * int(per_col_blank[c]) / 83, 1)}
                    for c in COLS if per_col_blank[c] > 0],
    columns_complete=[c for c in COLS if per_col_blank[c] == 0],
    per_site=dict(min=int(per_site_blank.min()), median=float(np.median(per_site_blank)),
                  mean=round(float(per_site_blank.mean()), 3), max=int(per_site_blank.max()),
                  sites_zero_blank=int((per_site_blank == 0).sum()),
                  sites_ge3_blank=int((per_site_blank >= 3).sum())),
    by_holder=[{k: (int(v) if isinstance(v, (np.integer,)) else
                    (round(float(v), 3) if isinstance(v, (float, np.floating)) else v))
                for k, v in r.items()} for r in by_holder.to_dict("records")],
    by_region=[{k: (int(v) if isinstance(v, (np.integer,)) else
                    (round(float(v), 3) if isinstance(v, (float, np.floating)) else v))
                for k, v in r.items()} for r in by_region.to_dict("records")],
    fake_nulls=dict(registration_year=RY, sky_quality_sqm=SQM, sky_quality_raw=RAWQ, contact=CONTACT),
    map_nulls=MAPNULL,
    cross_check_day14=dict(day14_sites_outside_outline=4,
                           day14_sites_without_own_cell_vs_day3_grid=7,
                           day14_sites_without_own_cell_vs_ne10m_fill=5,
                           today_outside_outline=MAPNULL["n_outside"],
                           today_no_own_cell_day3=MAPNULL["n_no_own_cell"],
                           today_no_own_cell_ne10m=MAPNULL["n_no_own_cell_vs_ne10m_fill"],
                           note="คำนวณใหม่จากทะเบียน + Natural Earth 10m + H3 res 5 แล้วเทียบกับ "
                                "Day 14 ทั้งสองนิยาม (กริด GADM ของ Day 3 ให้ 7 จุด "
                                "กริดที่เติมจาก NE 10m ให้ 5 จุด) — ตรงกันทุกตัว"),
    cross_check_day16=dict(source="day16_sites_osm.csv", zero_way_sites=MAPNULL["n_zero_way"],
                           note="จำนวน way ในรัศมี 1 กม. รอบจุด (นิยามเดียวกับ Day 16)"),
    figures=["day18_null.png", "day18_matrix.png", "day18_fake_nulls.png"],
    metric_definitions={
        "blank": "เซลล์ที่ค่า (หลังตัดช่องว่างหัวท้าย) เป็นสตริงว่าง — นับว่า *ไม่มีข้อมูล* ไม่ใช่ 0",
        "blank_fields": "จำนวนฟิลด์ (จาก 19) ที่ว่างของแต่ละจุด",
        "completeness_pct": "(19 − จำนวนฟิลด์ว่าง) ÷ 19 × 100 (ร้อยละ) ของแต่ละจุด",
        "registration_year.is_range": "จำนวนจุดที่ค่ามีเครื่องหมายช่วง (– หรือ -) ซึ่งหมายถึง *ระยะเวลาที่ใช้ได้*"
                                      " ไม่ใช่ปีที่ลงทะเบียนตามที่ชื่อคอลัมน์บอก",
        "sky_quality_sqm.integer_form / decimal_form": "จำนวนจุดที่ค่าเขียนเป็นจำนวนเต็ม เทียบกับทศนิยม "
                                                       "ในคอลัมน์เดียวกัน (ความละเอียดปนกัน)",
        "contact.duplicated_values": "จำนวนค่า contact ที่ถูกใช้ซ้ำในหลายจุด (ฟิลด์ที่ควรมีต่อจุด)",
        "outside_outline": "จุดที่พิกัดไม่อยู่ใน polygon ประเทศไทยของ Natural Earth 10m",
        "no_own_cell": "จุดที่เซลล์ H3 res 5 ของตัวเองไม่อยู่ในกริด 1,738 เซลล์ของ Day 3 "
                       "(เพราะจุดกลางเซลล์อยู่นอกประเทศ)",
        "zero_way_map": "จุดที่ไม่มี way ใดใน OpenStreetMap ที่จุดศูนย์กลางอยู่ภายใน 1 กม. (ต่อจาก Day 16)",
        "sites_null_in_two_or_more_senses": "จุดที่ว่างพร้อมกันตั้งแต่สองความหมายขึ้นไป "
                                           "(นอกเส้นขอบ / ไม่มี hex / ไม่มีวัตถุในแผนที่)"},
    note="ทะเบียนอ่านด้วย keep_default_na=False เพื่อให้ค่าว่างเป็นสตริงว่างจริง ๆ "
         "ไม่ถูก pandas แปลงเป็น NaN แล้วนับตกหล่น")
json.dump(STATS, open(os.path.join(OUT, "day18_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2,
          default=lambda o: o.item() if hasattr(o, "item") else str(o))
log("เขียน day18_stats.json + day18_null_sites.csv + day18_by_holder.csv")

## ตอน 7) referee

In [ ]:
# ---------------------------------------------------------------- 7. referee
for f in ["day18_null.png", "day18_matrix.png", "day18_fake_nulls.png",
          "day18_null_sites.csv", "day18_by_holder.csv", "day18_stats.json"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day18_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day18-texts")
assert int(blank.values.sum()) == int(per_site_blank.sum()) == int(per_col_blank.sum()), \
    "ผลรวมความว่างต้องตรงกันทั้งแบบต่อจุด ต่อคอลัมน์ และทั้งตาราง"
assert MAPNULL["n_outside"] == 4, "จุดนอกเส้นขอบประเทศต้องเป็น 4 ตามที่ Day 14 บันทึก"
assert MAPNULL["n_no_own_cell"] == 7, "จุดที่ไม่มี hex ในกริด Day 3 ต้องเป็น 7 ตามที่ Day 14 บันทึก (n_day3)"
assert MAPNULL["n_no_own_cell_vs_ne10m_fill"] == 5, \
    "จุดที่ไม่มี hex ในกริดที่เติมจาก NE 10m ต้องเป็น 5 ตามที่ Day 14 บันทึก (n_ne10m)"
assert len(MAPNULL["sites_null_in_two_or_more_senses"]) >= 1
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps(dict(cells_blank=int(blank.values.sum()),
                               blank_share_pct=STATS["register"]["blank_share_pct"],
                               sites_zero_blank=STATS["per_site"]["sites_zero_blank"],
                               ry_distinct=RY["distinct"], ry_range=RY["is_range"],
                               sqm_dec=SQM["decimal_form"], contact_dup=CONTACT["duplicated_values"],
                               outside=MAPNULL["n_outside"], no_cell=MAPNULL["n_no_own_cell"],
                               zero_way=MAPNULL["n_zero_way"], multi=MAPNULL["n_multi"]),
                          ensure_ascii=False))
print("DONE day18 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()